# Module 3: Imbalanced Machine Learning, Calibration & Profit Optimization
### Course: Applied Data Science & Machine Learning in Production
**Case Study:** Customer Churn & Lifetime Value (LTV) Prediction for a Subscription Business

---
### The Machine Learning Dilemma:
In real-world subscription churn, default classification thresholds ($t=0.50$) and naive accuracy are commercially disastrous:
1. **The Accuracy Paradox:** In an 80/20 imbalanced dataset, a model predicting "Nobody Churns" achieves 80% accuracy, while driving the company into bankruptcy.
2. **Uncalibrated Probabilities:** Tree-based ensembles often output distorted probabilities, making raw scores unreliable for expected value calculations.
3. **Asymmetric Business Payoffs:** The cost of an outreach email or small discount ($40) is minuscule compared to the loss of a churned customer ($600).

### Learning Objectives:
1. **Imbalanced Metrics:** Evaluate using **ROC-AUC**, **PR-AUC (Average Precision)**, and **Brier Score**.
2. **Probability Calibration:** Apply **Isotonic Regression** and **Platt Scaling** to ensure $P(\text{churn} = 1)$ accurately reflects real empirical frequencies.
3. **Expected Value Threshold Optimization:** Formulate the financial profit curve to select the optimal operational decision threshold $t^*$.
4. **Dual-Model Inference:** Pair binary churn probabilities with continuous Customer Lifetime Value (LTV) regression.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, precision_recall_curve, roc_curve
)
from sklearn.pipeline import Pipeline

import sys
sys.path.append("..")
from src.features import create_preprocessor
from src.data_loader import load_raw_data, temporal_train_test_split

sns.set_theme(style="whitegrid")
DATA_PATH = Path("../data/raw/subscriptions_raw.csv")
df = load_raw_data(DATA_PATH)
train_df, test_df = temporal_train_test_split(df, cutoff_quantile=0.80)

feature_cols = [c for c in train_df.columns if c not in ["customer_id", "signup_date", "churn", "customer_ltv"]]
X_train = train_df[feature_cols]
y_train_churn = train_df["churn"].values
y_train_ltv = train_df["customer_ltv"].values

X_test = test_df[feature_cols]
y_test_churn = test_df["churn"].values
y_test_ltv = test_df["customer_ltv"].values

print(f"Training on {len(X_train)} samples, testing on {len(X_test)} samples.")
print(f"Base Churn Rate in Test Set: {y_test_churn.mean():.2%}")

## 1. Training & Probability Calibration
We train a Gradient Boosting model and calibrate its predicted probabilities using Isotonic Regression via `CalibratedClassifierCV`.

In [ ]:
# SOLUTION: Train Calibrated Churn Pipeline
preprocessor = create_preprocessor()
base_model = GradientBoostingClassifier(n_estimators=100, max_depth=4, learning_rate=0.08, random_state=42)

# Probability Calibration wrapper
calibrated_clf = CalibratedClassifierCV(estimator=base_model, method="isotonic", cv=3)

churn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", calibrated_clf)
])

churn_pipeline.fit(X_train, y_train_churn)
y_probs = churn_pipeline.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test_churn, y_probs)
pr_auc = average_precision_score(y_test_churn, y_probs)
brier = brier_score_loss(y_test_churn, y_probs)

print(f"=== Churn Model Performance ===")
print(f"ROC-AUC:       {roc_auc:.4f}")
print(f"PR-AUC:        {pr_auc:.4f} (Baseline: {y_test_churn.mean():.4f})")
print(f"Brier Score:   {brier:.4f} (Closer to 0 is better calibrated)")

## 2. Evaluating Calibration: The Reliability Curve
A well-calibrated model means that among customers given a 70% churn probability, approximately 70% actually churn.

In [ ]:
prob_true, prob_pred = calibration_curve(y_test_churn, y_probs, n_bins=8, strategy="uniform")

plt.figure(figsize=(7, 6))
plt.plot(prob_pred, prob_true, marker="o", linewidth=2, label="Calibrated Model")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect Calibration")
plt.title("Reliability Diagram (Calibration Curve)")
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Empirical Fraction of Churners")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Business Cost-Benefit Matrix & Expected Value Optimization
Instead of defaulting to $t=0.50$, we optimize the decision threshold based on business economics:
- **Intervention Cost ($C$):** $40 (retention incentive/discount/outreach)
- **Customer Lifetime Value Saved ($V$):** $550 (average margin saved)
- **Retention Campaign Success Probability ($p_{\text{save}}$):** 50%

Payoffs:
- **True Positive (Intervene on actual churner):** $p_{\text{save}} \times V - C = (0.50 \times 550) - 40 = +\$235$
- **False Positive (Intervene on loyal customer):** $-C = -\$40$
- **False Negative (Missed churner):** $$0$ saved, lost future revenue
- **True Negative (Loyal customer left alone):** $$0$

In [ ]:
# SOLUTION: Financial Profit Curve & Threshold Optimization
intervention_cost = 40.0
avg_saved_value = 550.0
p_save = 0.50

thresholds = np.linspace(0.05, 0.95, 100)
net_profits = []

for t in thresholds:
    y_pred = (y_probs >= t).astype(int)
    tp = np.sum((y_test_churn == 1) & (y_pred == 1))
    fp = np.sum((y_test_churn == 0) & (y_pred == 1))
    
    profit = tp * (p_save * avg_saved_value - intervention_cost) - fp * intervention_cost
    net_profits.append(profit)

best_idx = np.argmax(net_profits)
optimal_threshold = thresholds[best_idx]
max_profit = net_profits[best_idx]
profit_at_default = net_profits[np.argmin(np.abs(thresholds - 0.50))]

print(f"Optimal Decision Threshold:  {optimal_threshold:.3f}")
print(f"Net Profit at Optimal Threshold: ${max_profit:,.2f}")
print(f"Net Profit at Default (0.50):    ${profit_at_default:,.2f}")
print(f"Commercial Profit Uplift:        ${max_profit - profit_at_default:,.2f}")

# Plotting Profit Curve
plt.figure(figsize=(9, 5))
plt.plot(thresholds, net_profits, color="darkgreen", lw=2, label="Net Profit Curve")
plt.axvline(optimal_threshold, color="red", linestyle="--", label=f"Optimal Threshold ({optimal_threshold:.2f})")
plt.axvline(0.50, color="gray", linestyle=":", label="Default Threshold (0.50)")
plt.title("Net Business Profit vs Decision Threshold")
plt.xlabel("Probability Decision Threshold")
plt.ylabel("Expected Net Profit ($)")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Continuous Lifetime Value (LTV) Regressor
We also train a continuous regressor to estimate dollar-value LTV, enabling tier-based prioritization.

In [ ]:
ltv_pipeline = Pipeline([
    ("preprocessor", create_preprocessor()),
    ("regressor", GradientBoostingRegressor(n_estimators=100, max_depth=4, learning_rate=0.08, random_state=42))
])

ltv_pipeline.fit(X_train, y_train_ltv)
ltv_preds = ltv_pipeline.predict(X_test)

mae = np.mean(np.abs(y_test_ltv - ltv_preds))
print(f"LTV Regressor Mean Absolute Error (MAE): ${mae:.2f}")

## 5. Course Synthesis: The Complete Production Pipeline
We have journeyed from raw subscription data to calibrated probabilistic modeling and financial expected-value decisioning:
- **Module 1:** DuckDB SQL for rapid cohort discovery.
- **Module 2:** Leak-free temporal engineering and custom Scikit-Learn transformers.
- **Module 3:** Imbalanced ML, probability calibration, and financial profit maximization.
- **Next Step:** Deploying to the **FastAPI** service (`src/app.py`) with automated CI/CD testing!